# Superinvestor overlap: stocks several top managers own

Combine several managers' 13F holdings to find the stocks they share and the ones more than one of them bought last quarter.

Read it with charts at https://www.quantisnow.com/sec-api/guides/superinvestor-overlap-13f. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

One fund buying a stock is an opinion. Three unrelated managers opening positions in the same quarter is a pattern worth a look. Pull each manager's holdings by CIK, then count.

## Every fund that opened a position in one stock

### Which funds opened a new position in Nvidia last quarter?

New positions are the clearest read of fresh conviction in a 13F, larger ones first.

```shell
curl -X POST https://www.quantisnow.com/api/v1/13f/holdings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "issuer.ticker:NVDA AND change:new AND period:2026-06-30",
  "from": 0,
  "size": 5,
  "sort": [{"valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 216):

```json
{
  "total": {
    "value": 216,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001313360-26-000003:1045810",
      "period": "2026-06-30",
      "filedOn": "2026-08-07",
      "accessionNo": "0001313360-26-000003",
      "manager": {
        "cik": "0001313360",
        "name": "SG Americas Securities, LLC",
        "city": "NEW YORK",
        "state": "NY"
      },
      "issuer": {
        "cik": "0001045810",
        "ticker": "NVDA",
        "name": "NVIDIA Corporation"
      },
      "shares": 62298007,
      "valueUsd": 12465208000,
      "previousShares": 0,
      "change": "new",
      "changeValueUsd": 12465208000
    },
    {
      "id": "0001213900-26-088240:1045810",
      "period": "2026-06-30",
      "filedOn": "2026-08-12",
      "accessionNo": "0001213900-26-088240",
      "manager": {
        "cik": "0001633516",
        "name": "NewEdge Advisors, LLC",
        "city": "NEW ORLEANS",
        "state": "LA"
      },
      "issuer": {
        "cik": "0001045810",
        "ticker": "NVDA",
        "name": "NVIDIA Corporation"
      },
      "shares": 3483179,
      "valueUsd": 696949263,
      "previousShares": 0,
      "change": "new",
      "changeValueUsd": 696949263
    }
  ]
}
```

## In Python

In [ ]:
import os, requests
from collections import defaultdict

CIKS = ["0001067983"]  # add the CIKs of the other managers you follow
H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}

owners, new = defaultdict(set), defaultdict(set)
for cik in CIKS:
    start = 0
    while True:
        page = requests.post("https://www.quantisnow.com/api/v1/13f/holdings", headers=H, json={
            "query": f"manager.cik:{cik} AND period:2026-06-30", "size": 50, "from": start,
        }).json()
        for h in page["data"]:
            t = h["issuer"]["ticker"]
            owners[t].add(h["manager"]["name"])
            if h["change"] == "new":
                new[t].add(h["manager"]["name"])
        start += 50
        if start >= page["total"]["value"]:
            break

for t, who in sorted(owners.items(), key=lambda kv: -len(kv[1]))[:20]:
    print(t, len(who), "own,", len(new[t]), "new")

This uses one call per 50 holdings per manager. For the reverse, every fund holding one stock, query `issuer.ticker:NVDA` instead, as in the example above.

Related: [Famous managers' new buys](https://www.quantisnow.com/sec-api/guides/famous-investors-13f-new-positions), [Which funds bought a stock](https://www.quantisnow.com/sec-api/guides/which-funds-bought-a-stock).